In [6]:
pip install yt-dlp pandas

Note: you may need to restart the kernel to use updated packages.


# Scraper tahap 1 Menggunakan keyword
Output: json platform, judul, link, creator, tanggal upload, views, like, jumlah komentar

In [8]:
import yt_dlp
import json
import os
from datetime import datetime

def scrape_youtube(keyword, max_results=5):
    # Konfigurasi yt-dlp
    ydl_opts = {
        'quiet': True,              # Mematikan log bawaan terminal agar rapi
        'extract_flat': False,      # False = Ambil metadata mendalam (likes, views, komentar)
        'ignoreerrors': True,       # Lanjut ke video berikutnya jika ada error
        'no_warnings': True
    }

    search_query = f"ytsearch{max_results}:{keyword}"
    timestamp = datetime.now().isoformat()
    data_list = []

    print(f"Mencari {max_results} video teratas untuk keyword: '{keyword}'...\n")

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        try:
            result = ydl.extract_info(search_query, download=False)
            
            if 'entries' in result:
                for index, video in enumerate(result['entries'], start=1):
                    if not video:
                        continue
                    
                    # yt-dlp mengembalikan format tanggal YYYYMMDD. Kita ubah ke YYYY-MM-DD
                    raw_date = video.get('upload_date', '')
                    if len(raw_date) == 8:
                        formatted_date = f"{raw_date[:4]}-{raw_date[4:6]}-{raw_date[6:]}"
                    else:
                        formatted_date = raw_date

                    # Format data disesuaikan dengan standar output
                    item = {
                        "scraped_at": timestamp,
                        "platform": "YouTube",
                        "judul": video.get('title'),
                        "related_link": video.get('webpage_url') or f"https://www.youtube.com/watch?v={video.get('id')}",
                        "creator": video.get('uploader'),
                        "tanggal_upload": formatted_date,
                        "views": video.get('view_count', 0),
                        "likes": video.get('like_count', 0),
                        "jumlah_komentar": video.get('comment_count', 0)
                    }
                    data_list.append(item)
                    print(f"[{index}/{max_results}] ✔️ Berhasil: {item['judul'][:50]}...")
                    
        except Exception as e:
            print(f"Error saat scraping: {e}")

    return data_list

# ==========================================
# EKSEKUSI SCRAPING
# ==========================================
keyword_pencarian = "kucing pidato"
jumlah_target = 10 

hasil = scrape_youtube(keyword_pencarian, max_results=jumlah_target)

# Simpan data ke JSON
if hasil:
    # Buat folder 'data' jika belum ada 
    target_dir = "data"
    os.makedirs(target_dir, exist_ok=True)
    
    # Format nama file: youtube_stand_up.json
    filename = os.path.join(target_dir, f"youtube_{keyword_pencarian.replace(' ', '_')}.json")
    
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(hasil, f, indent=4, ensure_ascii=False)
        
    print(f"\n✅ Scraping Selesai! {len(hasil)} data disimpan di: {filename}")
else:
    print("\n❌ Tidak ada data yang berhasil diambil.")

Mencari 10 video teratas untuk keyword: 'kucing pidato'...

[1/10] ✔️ Berhasil: KULTUM  MEMBAHAS MASALALU KUCING PETANI...
[2/10] ✔️ Berhasil: kucing pidato...
[3/10] ✔️ Berhasil: 5 MENIT YG MENGUBAH HIDUP...
[4/10] ✔️ Berhasil: Pandji Pragiwaksono Tuai Kontroversi Setelah Sebut...
[5/10] ✔️ Berhasil: Apakah Kita Akan Bertemu Hewan Peliharaan Di AKHIR...
[6/10] ✔️ Berhasil: Awal mula penciptaan kucing (Habib Abdurahman bin ...
[7/10] ✔️ Berhasil: kucing pidato 😨😱😱😰...
[8/10] ✔️ Berhasil: Kucing pidato #viralvidio #trending #kucing #funny...
[9/10] ✔️ Berhasil: Cinta dengan kucing peliharaan,,,? ustad adi hiday...
[10/10] ✔️ Berhasil: Saran Ustadz Dr Khalid Basalamah untuk Tidak Memel...

✅ Scraping Selesai! 10 data disimpan di: data\youtube_kucing_pidato.json


# Scarping Komentar berdasarkan link pada json sebelumnya
Output: json, Judul, komentar, nama akun yang komentar

In [ ]:
import json
import yt_dlp
import os

# 1. Tentukan lokasi file JSON hasil scraping video sebelumnya
# Pastikan nama filenya sesuai dengan yang Anda hasilkan di tahap sebelumnya
file_video_sumber = "data/youtube_kucing_pidato.json" 

def scrape_komentar(filepath):
    # Buka dan baca file JSON
    with open(filepath, "r", encoding="utf-8") as f:
        video_data = json.load(f)

    # Konfigurasi yt-dlp khusus untuk mengambil komentar
    ydl_opts_comments = {
        'quiet': True,
        'extract_flat': False,
        'skip_download': True,    # Wajib: Pastikan tidak mengunduh videonya
        'getcomments': True,      # Kunci utama untuk menarik komentar
        # Opsional: Membatasi hanya mengambil maksimal 15 komentar teratas per video agar cepat
        'extractor_args': {'youtube': {'comment_sort': ['top'], 'max_comments': ['15,all,all']}},
        'ignoreerrors': True,
        'no_warnings': True
    }

    hasil_komentar = []
    print(f"Memulai scraping komentar dari {len(video_data)} video...\n")

    with yt_dlp.YoutubeDL(ydl_opts_comments) as ydl:
        for index, video in enumerate(video_data, start=1):
            url = video.get('related_link')
            judul = video.get('judul')
            
            print(f"[{index}/{len(video_data)}] Menarik komentar: {judul[:40]}...")
            
            try:
                # Ekstraksi informasi video beserta komentarnya
                info = ydl.extract_info(url, download=False)
                
                # Mengambil list of dictionary komentar
                comments = info.get('comments', [])
                
                if not comments:
                    print("  -> ⚠️ Tidak ada komentar atau fitur komentar dinonaktifkan.")
                    continue
                
                # Parsing isi komentar
                for c in comments:
                    komentar_item = {
                        "video_sumber": judul,
                        "link_video": url,
                        "nama_channel": c.get('author'),
                        "isi_komentar": c.get('text')
                    }
                    hasil_komentar.append(komentar_item)
                
                print(f"  -> ✔️ Berhasil mengambil {len(comments)} komentar.")
                
            except Exception as e:
                print(f"  -> ❌ Error: {e}")

    return hasil_komentar

# ==========================================
# EKSEKUSI SCRAPING KOMENTAR
# ==========================================
if os.path.exists(file_video_sumber):
    data_komentar = scrape_komentar(file_video_sumber)

    # Simpan hasil komentar ke file JSON baruimport json
import yt_dlp
import os

# 1. Tentukan lokasi file JSON hasil scraping video sebelumnya
# Pastikan nama filenya sesuai dengan yang Anda hasilkan di tahap sebelumnya
file_video_sumber = "data/youtube_stand_up.json" 

def scrape_komentar(filepath):
    # Buka dan baca file JSON
    with open(filepath, "r", encoding="utf-8") as f:
        video_data = json.load(f)

    # Konfigurasi yt-dlp khusus untuk mengambil komentar
    ydl_opts_comments = {
        'quiet': True,
        'extract_flat': False,
        'skip_download': True,    # Wajib: Pastikan tidak mengunduh videonya
        'getcomments': True,      # Kunci utama untuk menarik komentar
        # Opsional: Membatasi hanya mengambil maksimal 15 komentar teratas per video agar cepat
        'extractor_args': {'youtube': {'comment_sort': ['top'], 'max_comments': ['15,all,all']}},
        'ignoreerrors': True,
        'no_warnings': True
    }

    hasil_komentar = []
    print(f"Memulai scraping komentar dari {len(video_data)} video...\n")

    with yt_dlp.YoutubeDL(ydl_opts_comments) as ydl:
        for index, video in enumerate(video_data, start=1):
            url = video.get('related_link')
            judul = video.get('judul')
            
            print(f"[{index}/{len(video_data)}] Menarik komentar: {judul[:40]}...")
            
            try:
                # Ekstraksi informasi video beserta komentarnya
                info = ydl.extract_info(url, download=False)
                
                # Mengambil list of dictionary komentar
                comments = info.get('comments', [])
                
                if not comments:
                    print("  -> ⚠️ Tidak ada komentar atau fitur komentar dinonaktifkan.")
                    continue
                
                # Parsing isi komentar
                for c in comments:
                    komentar_item = {
                        "video_sumber": judul,
                        "link_video": url,
                        "nama_channel": c.get('author'),
                        "isi_komentar": c.get('text')
                    }
                    hasil_komentar.append(komentar_item)
                
                print(f"  -> ✔️ Berhasil mengambil {len(comments)} komentar.")
                
            except Exception as e:
                print(f"  -> ❌ Error: {e}")

    return hasil_komentar

# ==========================================
# EKSEKUSI SCRAPING KOMENTAR
# ==========================================
if os.path.exists(file_video_sumber):
    data_komentar = scrape_komentar(file_video_sumber)

    # Simpan hasil komentar ke file JSON baru
    if data_komentar:
        output_file = "data/komentar_youtube_stand_up.json"
        
        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(data_komentar, f, indent=4, ensure_ascii=False)
            
        print(f"\n✅ Selesai! {len(data_komentar)} komentar berhasil disimpan di: {output_file}")
    else:
        print("\n❌ Tidak ada komentar yang berhasil diambil.")
else:
    print(f"File {file_video_sumber} tidak ditemukan. Pastikan Anda sudah menjalankan scraping tahap pertama.")
    if data_komentar:
        output_file = "data/komentar_youtube_stand_up.json"
        
        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(data_komentar, f, indent=4, ensure_ascii=False)
            
        print(f"\n✅ Selesai! {len(data_komentar)} komentar berhasil disimpan di: {output_file}")
    else:
        print("\n❌ Tidak ada komentar yang berhasil diambil.")
else:
    print(f"File {file_video_sumber} tidak ditemukan. Pastikan Anda sudah menjalankan scraping tahap pertama.")

Memulai scraping komentar dari 10 video...

[1/10] Menarik komentar: KULTUM  MEMBAHAS MASALALU KUCING PETANI...
  -> ✔️ Berhasil mengambil 1090 komentar.
[2/10] Menarik komentar: kucing pidato...
  -> ⚠️ Tidak ada komentar atau fitur komentar dinonaktifkan.
[3/10] Menarik komentar: 5 MENIT YG MENGUBAH HIDUP...
  -> ✔️ Berhasil mengambil 1993 komentar.
[4/10] Menarik komentar: Pandji Pragiwaksono Tuai Kontroversi Set...
  -> ✔️ Berhasil mengambil 374 komentar.
[5/10] Menarik komentar: Apakah Kita Akan Bertemu Hewan Peliharaa...
  -> ✔️ Berhasil mengambil 224 komentar.
[6/10] Menarik komentar: Awal mula penciptaan kucing (Habib Abdur...
  -> ✔️ Berhasil mengambil 116 komentar.
[7/10] Menarik komentar: kucing pidato 😨😱😱😰...
  -> ⚠️ Tidak ada komentar atau fitur komentar dinonaktifkan.
[8/10] Menarik komentar: Kucing pidato #viralvidio #trending #kuc...
  -> ⚠️ Tidak ada komentar atau fitur komentar dinonaktifkan.
[9/10] Menarik komentar: Cinta dengan kucing peliharaan,,,? ustad...
  -> ✔️

# Scarping download video berdasarkan link hasil scarp keyword

In [5]:
import json
import yt_dlp
import os

# 1. Lokasi file JSON dari tahap pertama
file_video_sumber = "data/youtube_kucing_pidato.json"

# 2. Folder tempat video akan disimpan
folder_output_video = "data/videos"
os.makedirs(folder_output_video, exist_ok=True)

def download_video_dari_json(filepath):
    # Baca data video dari JSON
    with open(filepath, "r", encoding="utf-8") as f:
        video_data = json.load(f)

    # Konfigurasi yt-dlp KHUSUS UNTUK DOWNLOAD
    ydl_opts_download = {
        # Ambil kualitas maksimal 720p agar ukuran tidak terlalu raksasa untuk prototype
        'format': 'bestvideo[height<=720]+bestaudio/best[height<=720]',
        
        # Format nama file hasil download (disimpan di folder data/videos/ dengan nama judul videonya)
        'outtmpl': f'{folder_output_video}/%(title)s.%(ext)s',
        
        # quiet diatur ke False agar Anda BISA MELIHAT progress bar persentase download di terminal
        'quiet': False, 
        'ignoreerrors': True,
        'no_warnings': True
    }

    print(f"Memulai proses download untuk {len(video_data)} video...\n")

    with yt_dlp.YoutubeDL(ydl_opts_download) as ydl:
        for index, video in enumerate(video_data, start=1):
            url = video.get('related_link')
            judul = video.get('judul')
            
            print(f"\n{'='*50}")
            print(f"[{index}/{len(video_data)}] Sedang mengunduh: {judul}")
            print(f"{'='*50}")
            
            try:
                # Perintah untuk mengunduh video
                ydl.download([url])
            except Exception as e:
                print(f"❌ Gagal mengunduh {judul}: {e}")

# ==========================================
# EKSEKUSI DOWNLOAD
# ==========================================
if os.path.exists(file_video_sumber):
    download_video_dari_json(file_video_sumber)
    print(f"\n✅ Proses unduh selesai! Video tersimpan di folder: {folder_output_video}")
else:
    print(f"File {file_video_sumber} tidak ditemukan.")

Memulai proses download untuk 10 video...


[1/10] Sedang mengunduh: KULTUM  MEMBAHAS MASALALU KUCING PETANI
[youtube] Extracting URL: https://www.youtube.com/watch?v=HSyl7BQyD3A
[youtube] HSyl7BQyD3A: Downloading webpage
[youtube] HSyl7BQyD3A: Downloading android vr player API JSON
[info] HSyl7BQyD3A: Downloading 1 format(s): 398+251
[download] Destination: data\videos\KULTUM  MEMBAHAS MASALALU KUCING PETANI.f398.mp4
[download] 100% of    7.15MiB in 00:00:03 at 2.01MiB/s   
[download] Destination: data\videos\KULTUM  MEMBAHAS MASALALU KUCING PETANI.f251.webm
[download] 100% of    1.07MiB in 00:00:00 at 1.25MiB/s   

[2/10] Sedang mengunduh: kucing pidato
[youtube] Extracting URL: https://www.youtube.com/watch?v=qM9ILi3ptxk
[youtube] qM9ILi3ptxk: Downloading webpage
[youtube] qM9ILi3ptxk: Downloading android vr player API JSON
[info] qM9ILi3ptxk: Downloading 1 format(s): 160+140
[download] Destination: data\videos\kucing pidato.f160.mp4
[download] 100% of  103.72KiB in 00:00:00 at 163.